# RAG Eval Starter
**Compare RAG configurations on your own documents. Find what actually moves the needle.**

Born from a real signal: builders consistently pick expensive models assuming cost = quality.  
It often doesn't. This notebook shows you what actually matters.

---

## What this notebook does
1. Loads your documents and splits them into chunks
2. Builds a FAISS vector index (local, no infra needed)
3. Tests **2 retrieval configurations** × **your choice of models**
4. Scores each configuration on 3 metrics: **Faithfulness**, **Answer Relevance**, **Context Precision**
5. Outputs a comparison table — so you can see what actually moved the needle

## Metrics explained (plain English)
| Metric | What it asks |
|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context, or is the model making things up? |
| **Answer Relevance** | Does the answer actually address the question asked? |
| **Context Precision** | Are the retrieved chunks actually useful, or is the model working around irrelevant noise? |

> All scores are 0–1. Higher is better.

In [ ]:
# Install dependencies if needed
# !pip install ragas langchain langchain-openai langchain-community faiss-cpu openai pandas python-dotenv pypdf

In [ ]:
import os
import json
import warnings
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

# ── Set your OpenAI API key ───────────────────────────────────────────────────
# Option 1: set OPENAI_API_KEY in a .env file in this folder
# Option 2: uncomment and paste directly (never commit this)
# os.environ['OPENAI_API_KEY'] = 'sk-...'

assert os.getenv('OPENAI_API_KEY'), "Set OPENAI_API_KEY in .env or above"
print("✓ API key loaded")

In [ ]:
# ── Configuration — edit these ────────────────────────────────────────────────

DOCS_DIR   = Path("data/sample_docs")    # folder containing .txt or .pdf files
QA_FILE    = Path("data/sample_qa.json") # list of {question, ground_truth} dicts

# Models to compare (any OpenAI chat model)
MODELS = [
    "gpt-4o-mini",   # cheap, fast
    "gpt-4o",        # expensive — does cost actually help here?
]

# Retrieval configurations to compare
CONFIGS = {
    "chunk_256_k3":  {"chunk_size": 256,  "chunk_overlap": 25,  "k": 3},
    "chunk_512_k5":  {"chunk_size": 512,  "chunk_overlap": 50,  "k": 5},
}

print(f"Docs dir:  {DOCS_DIR}")
print(f"QA pairs:  {QA_FILE}")
print(f"Models:    {MODELS}")
print(f"Configs:   {list(CONFIGS.keys())}")

In [ ]:
# ── Load documents ────────────────────────────────────────────────────────────

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def load_documents(docs_dir: Path) -> list:
    docs = []
    for f in docs_dir.iterdir():
        if f.suffix == '.txt':
            loader = TextLoader(str(f))
        elif f.suffix == '.pdf':
            loader = PyPDFLoader(str(f))
        else:
            continue
        docs.extend(loader.load())
    print(f"  Loaded {len(docs)} document(s) from {docs_dir}")
    return docs

raw_docs = load_documents(DOCS_DIR)

# Load QA pairs
with open(QA_FILE) as f:
    qa_pairs = json.load(f)

questions   = [qa['question']    for qa in qa_pairs]
ground_truth= [qa['ground_truth'] for qa in qa_pairs]
print(f"  Loaded {len(qa_pairs)} QA pairs")

In [ ]:
# ── Build retriever for a given config ───────────────────────────────────────

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

def build_retriever(docs: list, chunk_size: int, chunk_overlap: int, k: int):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)
    print(f"    {len(chunks)} chunks (size={chunk_size}, overlap={chunk_overlap})")

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    return vectorstore.as_retriever(search_kwargs={"k": k})


def run_rag(retriever, llm, questions: list) -> tuple[list, list]:
    """Run RAG for all questions. Returns (answers, contexts)."""
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        return_source_documents=True
    )
    answers, contexts = [], []
    for q in questions:
        result = chain.invoke({"query": q})
        answers.append(result['result'])
        contexts.append([doc.page_content for doc in result['source_documents']])
    return answers, contexts

print("✓ RAG pipeline functions ready")

In [ ]:
# ── Run all configurations and collect results ────────────────────────────────
# This cell makes API calls — costs pennies with gpt-4o-mini on the sample data.

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

results = []

for config_name, cfg in CONFIGS.items():
    print(f"\n── Config: {config_name} ──")
    retriever = build_retriever(
        raw_docs,
        chunk_size=cfg['chunk_size'],
        chunk_overlap=cfg['chunk_overlap'],
        k=cfg['k']
    )

    for model_name in MODELS:
        print(f"  Model: {model_name}")
        llm = ChatOpenAI(model=model_name, temperature=0)

        answers, contexts = run_rag(retriever, llm, questions)

        # Build RAGAS dataset
        ragas_data = Dataset.from_dict({
            "question":   questions,
            "answer":     answers,
            "contexts":   contexts,
            "ground_truth": ground_truth,
        })

        scores = evaluate(
            ragas_data,
            metrics=[faithfulness, answer_relevancy, context_precision],
            llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),  # evaluator model
        )

        results.append({
            "config":            config_name,
            "model":             model_name,
            "chunk_size":        cfg['chunk_size'],
            "top_k":             cfg['k'],
            "faithfulness":      round(scores['faithfulness'],      3),
            "answer_relevancy":  round(scores['answer_relevancy'],  3),
            "context_precision": round(scores['context_precision'], 3),
        })
        print(f"    faithfulness={results[-1]['faithfulness']} | relevancy={results[-1]['answer_relevancy']} | precision={results[-1]['context_precision']}")

print("\n✓ All configurations evaluated")

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────

df = pd.DataFrame(results)
df['avg_score'] = df[['faithfulness','answer_relevancy','context_precision']].mean(axis=1).round(3)
df = df.sort_values('avg_score', ascending=False)

print("\n=== RAG Configuration Comparison ===")
print(df[['config','model','faithfulness','answer_relevancy','context_precision','avg_score']].to_string(index=False))

# Save results
df.to_csv('results.csv', index=False)
print("\n✓ Saved to results.csv")

In [ ]:
# ── Verdict ───────────────────────────────────────────────────────────────────

best = df.iloc[0]
worst = df.iloc[-1]

print("\n=== VERDICT ===")
print(f"Best config:  {best['config']} + {best['model']}")
print(f"  avg score:  {best['avg_score']}")
print()
print(f"Worst config: {worst['config']} + {worst['model']}")
print(f"  avg score:  {worst['avg_score']}")
print()

gap = round(best['avg_score'] - worst['avg_score'], 3)
print(f"Performance gap: {gap}")

if gap > 0.1:
    print("→ Significant gap. Retrieval config matters more than you think.")
elif gap > 0.05:
    print("→ Moderate gap. Worth optimising retrieval before scaling model.")
else:
    print("→ Small gap on this dataset. Try with your own documents for a more discriminating result.")

# Cost observation
if 'gpt-4o' in df['model'].values and 'gpt-4o-mini' in df['model'].values:
    gpt4o_avg = df[df['model']=='gpt-4o']['avg_score'].mean()
    mini_avg  = df[df['model']=='gpt-4o-mini']['avg_score'].mean()
    print()
    print(f"gpt-4o avg score:      {round(gpt4o_avg, 3)}")
    print(f"gpt-4o-mini avg score: {round(mini_avg, 3)}")
    diff = round(gpt4o_avg - mini_avg, 3)
    if diff < 0.05:
        print(f"→ Cost difference: ~10x. Score difference: {diff}. The expensive model may not be worth it on this pipeline.")
    else:
        print(f"→ Score difference: {diff}. Evaluate whether the quality gain justifies the cost.")

---

## What to do next

1. **Replace sample docs** with your own documents in `data/sample_docs/`
2. **Replace sample QA** with questions your users actually ask in `data/sample_qa.json`
3. **Add more configurations** — try chunk sizes 128 and 1024, or add a reranker
4. **Read the scores critically** — RAGAS uses an LLM as judge, so treat scores as directional, not absolute

---

*Built from a real community signal. Source: [r/LocalLLaMA](https://www.reddit.com/r/LocalLLaMA)*  
*Part of the [ARTHA](https://github.com/SidharthKriplani/artha) open-source signal initiative.*